<a href="https://colab.research.google.com/github/Pedro-Soares09/PicPay-Open-Finance-Simulator/blob/main/Emprestimo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- INSTALAÇÃO ---
!pip install pyngrok flask-cors
from pyngrok import ngrok
from flask import Flask, request, jsonify
from flask_cors import CORS
from threading import Thread
import time
from datetime import datetime

# CONFIGURAÇÃO NGROK
NGROK_TOKEN = "3DY9c1cOhOFlvLbF9xPYCzmFlf3_3cR37Xtemmpv9jrwii34N" #COLOQUE SEU TOKEN AQUI
ngrok.set_auth_token(NGROK_TOKEN)

app = Flask(__name__)
CORS(app) # CRÍTICO: Permite que navegadores e outras linguagens acessem a API

# Banco de dados na memória
banco_central_dados = {}

@app.route('/central', methods=['GET'])
def buscar():
    print(f"[{datetime.now().strftime('%H:%M:%S')}] 📥 Consulta de dados recebida.")
    return jsonify(banco_central_dados)

@app.route('/central', methods=['POST'])
def salvar():
    global banco_central_dados
    banco_central_dados = request.json
    print(f"[{datetime.now().strftime('%H:%M:%S')}] 📤 Dados atualizados por agência externa!")
    return jsonify({"status": "sucesso", "total_registros": len(banco_central_dados)})

# Rodar servidor
def run():
    app.run(port=5000)

Thread(target=run).start()
time.sleep(2)
public_url = ngrok.connect(5000).public_url

print("\n" + "="*60)
print("🏦 BARRAMENTO OPEN FINANCE - BANCO CENTRAL ATIVO")
print("="*60)
print(f"\nLINK UNIVERSAL:")
print(f"👉 {public_url}/central")
print("\n" + "="*60)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from datetime import datetime, timedelta
import requests

# --- CONFIGURAÇÃO DA INTERCONEXÃO ---
URL_BANCO_CENTRAL = "https://mossy-bribe-carless.ngrok-free.dev/central"

# --- INTERFACE BANCÁRIA  ---
display(HTML("""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@300;400;600;700&display=swap');

    .app-container {
        background: #0f0f0f;
        font-family: 'Plus Jakarta Sans', sans-serif;
        border-radius: 20px;
        padding: 30px;
        color: #f0f0f0;
        max-width: 600px;
        border: 1px solid #2a2a2a;
        margin: auto;
        box-shadow: 0 30px 60px rgba(0,0,0,0.7);
    }

    .header-system {
        border-bottom: 2px solid #21C25E;
        padding-bottom: 15px;
        margin-bottom: 25px;
    }

    .picpay-green { color: #21C25E; }

    .widget-label { color: #888 !important; font-weight: 600 !important; }
    .widget-text input, .widget-floattext input {
        background: #1a1a1a !important;
        color: white !important;
        border: 1px solid #333 !important;
        border-radius: 8px !important;
        padding: 12px !important;
        width: 100% !important;
    }

    .btn-action {
        background: #21C25E !important;
        color: white !important;
        font-weight: 700 !important;
        border-radius: 10px !important;
        border: none !important;
        height: 50px !important;
        margin-top: 20px !important;
        cursor: pointer;
        text-transform: uppercase;
        letter-spacing: 1px;
    }

    .card-full {
        background: #181818;
        border-radius: 15px;
        padding: 20px;
        margin: 15px 0;
        border: 1px solid #2a2a2a;
    }

    .grid-info {
        display: grid;
        grid-template-columns: 1fr 1fr;
        gap: 10px;
        font-size: 0.85em;
        color: #aaa;
        margin-top: 10px;
    }

    .status-badge {
        padding: 4px 10px;
        border-radius: 5px;
        font-size: 10px;
        font-weight: 800;
        float: right;
    }

    .banner-msg {
        padding: 15px;
        border-radius: 10px;
        margin-top: 15px;
        text-align: center;
        font-weight: bold;
    }
</style>
"""))

class PicPayInterfaceCompletaV7:
    def __init__(self):
        self.out = widgets.Output()
        self.tabs = widgets.Tab(children=[self.ui_cadastro(), self.ui_extrato(), self.ui_renegociar()])
        self.tabs.set_title(0, '🏦 Novo Crédito')
        self.tabs.set_title(1, '📄 Open Finance')
        self.tabs.set_title(2, '🤝 Acordos')

    def baixar_dados(self):
        try:
            return requests.get(URL_BANCO_CENTRAL, headers={"ngrok-skip-browser-warning": "true"}, timeout=5).json()
        except:
            return {}

    def salvar_dados(self, dados):
        try:
            requests.post(URL_BANCO_CENTRAL, json=dados, timeout=5)
        except:
            with self.out:
                print("❌ Erro de Sincronização com o servidor")

    def formatar_cpf(self, change):
        nums = "".join(filter(str.isdigit, change['new']))
        fmt = f"{nums[:3]}.{nums[3:6]}.{nums[6:9]}-{nums[9:11]}" if len(nums) > 9 else nums
        change.owner.value = fmt[:14]

    def ui_cadastro(self):
        self.in_nome = widgets.Text(placeholder="Ex: João Silva", description="Nome:")
        self.in_mae = widgets.Text(placeholder="Ex: Maria Silva", description="Mãe:")
        self.in_cpf = widgets.Text(placeholder="000.000.000-00", description="CPF:")
        self.in_cpf.observe(self.formatar_cpf, names='value')
        self.in_end = widgets.Text(placeholder="Rua, Número, Bairro", description="Endereço:")
        self.in_renda = widgets.FloatText(description="Renda R$:")
        self.in_valor = widgets.FloatText(description="Solicitado:")

        btn = widgets.Button(description="EFETIVAR CONTRATAÇÃO", layout={'width': '100%'})
        btn.add_class("btn-action")
        btn.on_click(self.processar_contratacao)

        return widgets.VBox([
            widgets.HTML("<b style='color:#888'>DADOS DO PROPONENTE</b>"),
            self.in_nome, self.in_mae, self.in_cpf, self.in_end,
            widgets.HTML("<br><b style='color:#888'>ANÁLISE FINANCEIRA</b>"),
            self.in_renda, self.in_valor, btn
        ])

    def processar_contratacao(self, b):
        with self.out:
            clear_output()
            banco = self.baixar_dados()
            cpf_limpo = "".join(filter(str.isdigit, self.in_cpf.value))

            if len(cpf_limpo) != 11:
                display(HTML("<div class='banner-msg' style='background:#411; color:#f55'>CPF INVÁLIDO</div>"))
                return

            existente = banco.get(cpf_limpo)
            if existente and existente.get('status') == 'NEGATIVADO':
                origem = existente.get('origem', 'Outra Instituição')
                display(HTML(f"<div class='banner-msg' style='background:#411; color:#f55'>❌ REPROVADO: RESTRIÇÃO NO {origem.upper()}</div>"))
                return

            venc = (datetime.now() + timedelta(minutes=1)).strftime("%Y-%m-%d %H:%M:%S")
            banco[cpf_limpo] = {
                'nome': self.in_nome.value.upper(),
                'mae': self.in_mae.value.upper(),
                'end': self.in_end.value.upper(),
                'renda': float(self.in_renda.value),
                'valor': float(self.in_valor.value),
                'status': 'ATIVO',
                'score': 850,
                'vencimento': venc,
                'origem': 'PicPay'
            }
            self.salvar_dados(banco)
            display(HTML("<div class='banner-msg' style='background:#131; color:#21C25E'>✅ EMPRÉSTIMO AVERBADO COM SUCESSO!</div>"))

    def renderizar_extrato(self, b):
        with self.out_extrato:
            clear_output()
            banco = self.baixar_dados()
            agora = datetime.now()

            if not banco:
                print("Banco de dados vazio.")
                return

            for cpf, info in banco.items():
                if not isinstance(info, dict): continue

                status = info.get('status', 'ATIVO')
                origem = info.get('origem', 'Externo')
                valor = info.get('valor', 0.0)

                if origem == 'PicPay' and status == 'ATIVO':
                    try:
                        venc = datetime.strptime(info['vencimento'], "%Y-%m-%d %H:%M:%S")
                        if agora > venc:
                            info.update({'status': 'NEGATIVADO', 'valor': valor * 1.30, 'score': 150})
                            status, valor = 'NEGATIVADO', info['valor']
                            self.salvar_dados(banco)
                    except: pass

                cor = "#21C25E" if status == "ATIVO" else "#FF4B4B"

                display(HTML(f"""
                <div class='card-full'>
                    <span class='status-badge' style='background:{cor}22; color:{cor}'>{status}</span>
                    <b style='font-size:1.1em'>{info.get('nome', 'NÃO INFORMADO')}</b><br>
                    <small class='picpay-green'>Origem: {origem}</small>
                    <div class='grid-info'>
                        <span>CPF: {cpf}</span>
                        <span>Mãe: {info.get('mae', 'N/A')}</span>
                        <span>Endereço: {info.get('end', 'N/A')}</span>
                        <span>Score: {info.get('score', 0)}</span>
                    </div>
                    <div style='margin-top:15px; font-size:1.3em; font-weight:bold; color:{cor}'>
                        R$ {valor:.2f}
                    </div>
                </div>
                """))

    def ui_extrato(self):
        self.out_extrato = widgets.Output()
        btn = widgets.Button(description="CONSULTAR BASE DE DADOS", layout={'width': '100%'})
        btn.add_class("btn-action")
        btn.on_click(self.renderizar_extrato)
        return widgets.VBox([btn, self.out_extrato])

    def ui_renegociar(self):
        self.in_cpf_busca = widgets.Text(placeholder="CPF do devedor", description="CPF:")
        self.out_rec = widgets.Output()
        btn = widgets.Button(description="BUSCAR DÍVIDAS ATIVAS", layout={'width': '100%'})
        btn.add_class("btn-action")
        btn.on_click(self.buscar_divida)
        return widgets.VBox([self.in_cpf_busca, btn, self.out_rec])

    def buscar_divida(self, b):
        with self.out_rec:
            clear_output()
            banco = self.baixar_dados()
            cpf_alvo = "".join(filter(str.isdigit, self.in_cpf_busca.value))

            for cpf_b, info in banco.items():
                if "".join(filter(str.isdigit, cpf_b)) == cpf_alvo:
                    if info.get('status') == 'NEGATIVADO' and info.get('origem') == 'PicPay':
                        v_original = info['valor']
                        v_acordo = v_original * 0.70
                        display(HTML(f"""
                            <div class='card-full' style='border-color:#21C25E'>
                                <h3 class='picpay-green'>Oportunidade de Acordo</h3>
                                <span>Valor com Juros: R$ {v_original:.2f}</span><br>
                                <b style='font-size:1.5em; color:#21C25E'>Pague apenas: R$ {v_acordo:.2f}</b>
                            </div>
                        """))
                        btn_p = widgets.Button(description="QUITAR AGORA (30% OFF)", layout={'width': '100%'})
                        btn_p.add_class("btn-action")
                        btn_p.on_click(lambda x, c=cpf_b: self.quitar(c, banco))
                        display(btn_p)
                        return
            print("Nenhuma pendência PicPay encontrada.")

    def quitar(self, cpf, banco):
        banco[cpf].update({'status': 'ATIVO', 'valor': 0.0, 'score': 950, 'vencimento': '2099-01-01 00:00:00'})
        self.salvar_dados(banco)
        with self.out_rec:
            clear_output()
            display(HTML("<div class='banner-msg' style='background:#131; color:#21C25E'>DÍVIDA LIQUIDADA! NOME LIMPO NO OPEN FINANCE.</div>"))

# Iniciar
instancia = PicPayInterfaceCompletaV7()
display(HTML("""
<div class='app-container'>
    <div class='header-system'>
        <h1 style='margin:0'>PicPay<span class='picpay-green'>Bank</span></h1>
        <small style='color:#555'>SISTEMA DE GESTÃO DE CRÉDITO E OPEN FINANCE</small>
    </div>
"""), instancia.tabs, instancia.out, HTML("</div>"))